# EQO notebook: Hamiltonian to Trotter circuit

Create a typed Pauli-Hamiltonian input, submit EQO's published OpenQEvo Trotter workflow, then inspect the generated OpenQASM circuit and provenance report. OpenQEvo and Qiskit execute only in EQO's admitted environment; this notebook is a control-plane client.

Before running the notebook, start the full local profile from the repository root:

```bash
python -m pip install -e ".[local,jupyter]"
eqo local up
```

In [ ]:
import os

from eqo import EQOClient, render_artifact, render_run

EQO_ENDPOINT = os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080")
eqo = EQOClient.connect(EQO_ENDPOINT)
eqo.health()

## Select the published circuit-synthesis workflow

The published workflow fixes its scientifically reviewed configuration: the circuit-producing `qiskit_trotter` adapter, evolution time 1.0, four product-formula steps, and second-order Suzuki formula. To use another approved configuration, create a workflow revision in Compose and submit that published revision by its ID and version.

In [ ]:
workflow = next(
    (item for item in eqo.workflows.list() if item["id"] == "openqevo-trotter-synthesis"),
    None,
)
if workflow is None:
    raise RuntimeError("The OpenQEvo Trotter workflow is not published by this EQO profile.")
workflow

## Create the Hamiltonian input artifact

The `pauli` strings use Qiskit-compatible qubit ordering and every coefficient is real. The typed artifact preserves the exact input used by the workflow.

In [ ]:
import json

hamiltonian = {
    "qubits": 2,
    "terms": [
        {"pauli": "ZI", "coefficient": 1.0},
        {"pauli": "IZ", "coefficient": 0.5},
        {"pauli": "XX", "coefficient": 0.25},
    ],
}

input_hamiltonian = eqo.artifacts.create_input(
    "qhpc.pauli-hamiltonian@1",
    json.dumps(hamiltonian, indent=2),
    name="two-qubit-hamiltonian.json",
    labels={"example": "openqevo-trotter"},
)
render_artifact(input_hamiltonian)

## Submit and wait for the run

Submission is explicit. The SDK does not start containers, pull images, or resolve any external credential.

In [ ]:
run = eqo.workflows.submit(
    workflow["id"],
    workflow["version"],
    inputs={"hamiltonian": input_hamiltonian.id},
)
render_run(run)

In [ ]:
completed = run.wait(timeout=300)
if completed.state != "succeeded":
    raise RuntimeError(f"OpenQEvo synthesis ended in {completed.state}; inspect render_run(completed).")
render_run(completed)

## Inspect the generated circuit and provenance

The circuit is OpenQASM 2.0 in the workflow's supported basis. The separate report records the method parameters, OpenQEvo source revision, circuit depth, and gate counts.

In [ ]:
circuit = completed.artifacts.by_type("qhpc.quantum-circuit@1")
synthesis_report = completed.artifacts.by_type("qhpc.evolution-synthesis-report@1")

print(circuit.read_text())
render_artifact(synthesis_report)